In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from model_service.config import ModelServiceConfig
from model_service.preprocess.dataset_builder import build_pcam_datasets

cfg = ModelServiceConfig()

# Print out the experiment name and key configuration details for verification
print(f"🚀 Experiment: {cfg.model_name}")
# Check if artifacts directory is set correctly
print(f"Artifacts will be saved to: {cfg.paths.artifacts_checkpoints}")
# Check if project root is set correctly
print(f"Project Root: {cfg.paths.repo_root}")
# Check if Metal (GPU) is actually registered
print("Physical Devices:", tf.config.list_physical_devices())

Artifacts will be saved to: /Users/shayan/code/sandinosaso/pathsight/artifacts/checkpoints
Project Root: /Users/shayan/code/sandinosaso/pathsight
Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Load 1000 training samples
train_ds_test, val_ds, _, _ = build_pcam_datasets(max_train_samples=1000)

# Manually slice validation for the smoke test (to avoid function argument errors)
val_ds_test = val_ds.take(10)

print(f"Data ready. Input Shape: {cfg.data.input_shape}")

Data ready. Input Shape: [96, 96, 3]


In [ ]:
def build_convnext_model(cfg):
    # 1. Backbone: Pre-trained on ImageNet
    base_model = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights='imagenet',
        input_shape=tuple(cfg.data.input_shape)
    )
    base_model.trainable = False # Freeze weights for Stage 1

    # 2. Architecture: Functional API
    inputs = layers.Input(shape=tuple(cfg.data.input_shape))

    # Preprocessing is built-in to ConvNeXt in Keras 3
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)

    # 3. Compile using your Project Metrics
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.train.learning_rate),
        loss='binary_crossentropy',
        metrics=cfg.train.metrics
    )
    return model

In [17]:
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 3, 3, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,115,813 (107.25 MB)

 Trainable params: 98,561 (385.00 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

 Optimizer params: 197,124 (770.02 KB)

In [13]:
from model_service.config import ModelServiceConfig
import tensorflow as tf

cfg = ModelServiceConfig()

# Check if Metal (GPU) is actually registered
print("Physical Devices:", tf.config.list_physical_devices())

Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Force BOTH creation and fit to the CPU to avoid 'Variable on different device' errors
with tf.device('/CPU:0'):
    print("Building model on CPU to avoid M1 driver bugs...")
    model = build_convnext_model(cfg)

    print("Running 1-epoch Smoke Test...")
    model.fit(
        train_ds_test,
        validation_data=val_ds_test,
        epochs=1,
        steps_per_epoch=2
    )
    print("✅ Logic verified. No shape or metric errors.")

Building model on CPU to avoid M1 driver bugs...
Running 1-epoch Smoke Test...


Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
Proportion of examples rejected by sampler is high: [0.5][0.5 0.5][0 1]
I0000 00:00:1777037958.620649 4231664 service.cc:145] XLA service 0x8ad0bf200 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777037958.621741 4231664 service.cc:153]   StreamExecutor device (0): Host, Default Version
I0000 00:

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5312 - auc: 0.5935 - loss: 0.8741 - precision: 0.8167 - recall: 0.2933  

Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]
Proportion of examples rejected by sampler is high: [nan][0.442307681 0.557692289][nan nan]


2/2 ━━━━━━━━━━━━━━━━━━━━ 36s 22s/step - accuracy: 0.5417 - auc: 0.5992 - loss: 0.8509 - precision: 0.7556 - recall: 0.3715 - val_accuracy: 0.5450 - val_auc: 0.7457 - val_loss: 0.8577 - val_precision: 0.5450 - val_recall: 1.0000


2026-04-24 17:39:43.437582: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/Users/shayan/.pyenv/versions/3.10.6/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


✅ Logic verified. No shape or metric errors.
